In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv('/content/diabetes.csv')

print("Data Awal:")
print(df.head())

Data Awal:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [ ]:
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values.reshape(-1, 1)

print("\nShape X :", X.shape)
print("Shape y :", y.shape)


Shape X : (768, 8)
Shape y : (768, 1)


Pada kode ANN tanpa normalisasi, data dipisahkan menjadi data input dan target menggunakan kode X = df.drop('Outcome', axis=1).values dan y = df['Outcome'].values.reshape(-1, 1). Variabel X berisi seluruh fitur sebagai input model, sedangkan y berisi kolom Outcome sebagai target prediksi. Data kemudian diubah ke bentuk array NumPy agar dapat diproses oleh neural network. Namun, karena data langsung digunakan tanpa proses normalisasi, setiap fitur memiliki rentang nilai yang berbeda sehingga proses pembelajaran model menjadi kurang stabil. Hal ini menyebabkan penurunan error berjalan lambat dan akurasi model hanya sekitar 65%, sehingga model masih kurang baik dalam mendeteksi pasien diabetes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nData Train :", X_train.shape)
print("Data Test  :", X_test.shape)


Data Train : (614, 8)
Data Test  : (154, 8)


In [ ]:
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)

    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }

    return parameters

In [ ]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(int)

In [ ]:
def forward_propagation(X, parameters):

    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]

    Z1 = np.dot(W1, X.T) + b1
    A1 = relu(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }

    return A2, cache


In [ ]:
def compute_cost(A2, y):

    m = y.shape[0]

    cost = -1/m * np.sum(
        y.T * np.log(A2 + 1e-8) +
        (1 - y.T) * np.log(1 - A2 + 1e-8)
    )

    return cost

In [ ]:
def backward_propagation(X, y, parameters, cache):

    m = X.shape[0]

    W2 = parameters["W2"]

    A1 = cache["A1"]
    A2 = cache["A2"]
    Z1 = cache["Z1"]

    dZ2 = A2 - y.T
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * relu_derivative(Z1)

    dW1 = (1/m) * np.dot(dZ1, X)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }

    return grads



In [ ]:
def update_parameters(parameters, grads, learning_rate):

    parameters["W1"] = parameters["W1"] - learning_rate * grads["dW1"]
    parameters["b1"] = parameters["b1"] - learning_rate * grads["db1"]

    parameters["W2"] = parameters["W2"] - learning_rate * grads["dW2"]
    parameters["b2"] = parameters["b2"] - learning_rate * grads["db2"]

    return parameters


In [ ]:
def train_ann(
    X_train,
    y_train,
    hidden_size=8,
    epochs=5000,
    learning_rate=0.0001 #Learning rate dibuat kecil karena data belum dinormalisasi sehingga training kurang stabil.
):

    input_size = X_train.shape[1]
    output_size = 1

    parameters = initialize_parameters(
        input_size,
        hidden_size,
        output_size
    )
    for i in range(epochs):

        # FORWARD
        A2, cache = forward_propagation(X_train, parameters)

        # COST
        cost = compute_cost(A2, y_train)

        # BACKWARD
        grads = backward_propagation(
            X_train,
            y_train,
            parameters,
            cache
        ) # UPDATE
        parameters = update_parameters(
            parameters,
            grads,
            learning_rate
        )

        # PRINT COST
        if i % 500 == 0:
            print(f"Epoch {i} | Cost : {cost:.6f}")

    return parameters

In [ ]:
parameters = train_ann(
    X_train,
    y_train,
    hidden_size=8,
    epochs=5000,
    learning_rate=0.0001
)


Epoch 0 | Cost : 0.693874
Epoch 500 | Cost : 0.691924
Epoch 1000 | Cost : 0.690557
Epoch 1500 | Cost : 0.688905
Epoch 2000 | Cost : 0.685238
Epoch 2500 | Cost : 0.676747
Epoch 3000 | Cost : 0.668681
Epoch 3500 | Cost : 0.665017
Epoch 4000 | Cost : 0.662532
Epoch 4500 | Cost : 0.660195


In [ ]:
def predict(X, parameters):

    A2, _ = forward_propagation(X, parameters)

    predictions = (A2 > 0.5).astype(int)

    return predictions.T

In [ ]:
all_predictions = predict(X, parameters)

# Membuat dataframe hasil prediksi
hasil_prediksi = df.copy()

hasil_prediksi['Prediksi'] = all_predictions

# Mengubah angka menjadi keterangan
hasil_prediksi['Keterangan Prediksi'] = hasil_prediksi['Prediksi'].map({
    0: 'Tidak Diabetes',
    1: 'Diabetes'
})

hasil_prediksi['Keterangan Asli'] = hasil_prediksi['Outcome'].map({
    0: 'Tidak Diabetes',
    1: 'Diabetes'
})

print("\n==============================")
print("HASIL PREDIKSI SELURUH DATA")
print("==============================")

print(hasil_prediksi)



HASIL PREDIKSI SELURUH DATA
     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0              6      148             72             35        0  33.6   
1              1       85             66             29        0  26.6   
2              8      183             64              0        0  23.3   
3              1       89             66             23       94  28.1   
4              0      137             40             35      168  43.1   
..           ...      ...            ...            ...      ...   ...   
763           10      101             76             48      180  32.9   
764            2      122             70             27        0  36.8   
765            5      121             72             23      112  26.2   
766            1      126             60              0        0  30.1   
767            1       93             70             31        0  30.4   

     DiabetesPedigreeFunction  Age  Outcome  Prediksi Keterangan Prediksi  \
0    

In [ ]:
accuracy = accuracy_score(y, all_predictions)

print("\n======================")
print("HASIL EVALUASI MODEL")
print("======================")

print("\nAccuracy :")
print(accuracy)

print("\nConfusion Matrix :")
print(confusion_matrix(y, all_predictions))

print("\nClassification Report :")
print(classification_report(y, all_predictions))


HASIL EVALUASI MODEL

Accuracy :
0.6510416666666666

Confusion Matrix :
[[499   1]
 [267   1]]

Classification Report :
              precision    recall  f1-score   support

           0       0.65      1.00      0.79       500
           1       0.50      0.00      0.01       268

    accuracy                           0.65       768
   macro avg       0.58      0.50      0.40       768
weighted avg       0.60      0.65      0.52       768



In [ ]:
# =========================
# CONTOH PREDIKSI SEMUA DATA
# =========================
print("\n======================")
print("PREDIKSI SEMUA DATA")
print("======================")

for i in range(len(X)):

    # Ambil 1 data
    sample_data = X[i].reshape(1, -1)

    # Prediksi
    prediction = predict(sample_data, parameters)

    print(f"\nData ke-{i+1}")
    print("Data :", sample_data)

    print("Prediksi :", prediction[0][0])

    if prediction[0][0] == 1:
        print("Keterangan : Pasien Terindikasi Diabetes")
    else:
        print("Keterangan : Pasien Tidak Terindikasi Diabetes")


PREDIKSI SEMUA DATA

Data ke-1
Data : [[  6.    148.     72.     35.      0.     33.6     0.627  50.   ]]
Prediksi : 0
Keterangan : Pasien Tidak Terindikasi Diabetes

Data ke-2
Data : [[ 1.    85.    66.    29.     0.    26.6    0.351 31.   ]]
Prediksi : 0
Keterangan : Pasien Tidak Terindikasi Diabetes

Data ke-3
Data : [[  8.    183.     64.      0.      0.     23.3     0.672  32.   ]]
Prediksi : 0
Keterangan : Pasien Tidak Terindikasi Diabetes

Data ke-4
Data : [[ 1.    89.    66.    23.    94.    28.1    0.167 21.   ]]
Prediksi : 0
Keterangan : Pasien Tidak Terindikasi Diabetes

Data ke-5
Data : [[  0.    137.     40.     35.    168.     43.1     2.288  33.   ]]
Prediksi : 0
Keterangan : Pasien Tidak Terindikasi Diabetes

Data ke-6
Data : [[  5.    116.     74.      0.      0.     25.6     0.201  30.   ]]
Prediksi : 0
Keterangan : Pasien Tidak Terindikasi Diabetes

Data ke-7
Data : [[ 3.    78.    50.    32.    88.    31.     0.248 26.   ]]
Prediksi : 0
Keterangan : Pasien Tidak Te

Pada kode ANN tanpa normalisasi, data langsung digunakan untuk proses training tanpa dilakukan scaling terlebih dahulu. Hal ini menyebabkan setiap fitur memiliki rentang nilai yang berbeda sehingga proses pembelajaran model menjadi kurang stabil. Selama training, nilai error menurun cukup lambat dan model kesulitan mengenali pola data secara optimal.

Hasil pengujian menunjukkan bahwa akurasi model masih sekitar 65%, sehingga performanya belum terlalu baik. Dari confusion matrix dan classification report terlihat bahwa model lebih sering memprediksi data sebagai “Tidak Diabetes” dan kurang mampu mendeteksi pasien diabetes dengan benar. Hal ini menunjukkan bahwa tanpa normalisasi, performa Artificial Neural Network menjadi kurang optimal karena perbedaan skala data mempengaruhi proses pembelajaran model.